# Dimension 3.4 - Tailored Recommendations per Quadrant
Cluster-specific analysis, best-practice benchmarking, policy transfer feasibility.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.spatial.distance import mahalanobis

from hdi.config import *
from hdi.data.loaders import load_health_inputs, load_health_outcomes
from hdi.models.optimization import dea_efficiency
from hdi.models.clustering import classify_quadrants

## 1. Cluster-Specific Regression

In [ ]:
# Load data and classify quadrants
inputs_df = load_health_inputs()
outcomes_df = load_health_outcomes()

input_cols = ["health_exp_per_capita", "physicians_per_1000", "hospital_beds_per_1000"]
dea_scores = dea_efficiency(
    inputs=inputs_df[input_cols],
    outputs=outcomes_df[["life_expectancy"]]
)
quadrants = classify_quadrants(
    input_level=inputs_df[input_cols].mean(axis=1),
    efficiency=dea_scores
)

# Run separate OLS regressions per quadrant to find binding constraints
regression_results = {}
for q in ["Q1", "Q2", "Q3", "Q4"]:
    mask = quadrants == q
    X = inputs_df.loc[mask, input_cols]
    y = outcomes_df.loc[mask, "life_expectancy"]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    regression_results[q] = model
    print(f"\n=== {q} (n={mask.sum()}) ===")
    print(f"R-squared: {model.rsquared:.3f}")
    print("Significant predictors (p < 0.05):")
    sig = model.pvalues[model.pvalues < 0.05]
    for var, pval in sig.items():
        if var != "const":
            print(f"  {var}: coef={model.params[var]:.4f}, p={pval:.4f}")

## 2. Best-Practice Analysis (Q2 Countries)

In [ ]:
# Q2 countries: Low-input, High-efficiency -- what makes them successful?
q2_mask = quadrants == "Q2"
q2_countries = quadrants[q2_mask].index.tolist()

print(f"Q2 best-practice countries ({len(q2_countries)}):")
print(", ".join(q2_countries))
print()

# Compare Q2 vs rest on key dimensions
all_data = pd.concat([inputs_df[input_cols], outcomes_df[["life_expectancy"]]], axis=1)
all_data["quadrant"] = quadrants

q2_profile = all_data[all_data["quadrant"] == "Q2"].describe().T[["mean", "std"]]
rest_profile = all_data[all_data["quadrant"] != "Q2"].describe().T[["mean", "std"]]

comparison = pd.DataFrame({
    "Q2_mean": q2_profile["mean"],
    "Rest_mean": rest_profile["mean"],
    "ratio": q2_profile["mean"] / rest_profile["mean"]
})

print("Q2 vs Rest-of-World profile:")
print(comparison.to_string())
print()
print("Key insight: Q2 countries achieve comparable/better outcomes")
print("with significantly lower inputs, suggesting structural efficiency gains.")

## 3. Policy Transfer Feasibility (Mahalanobis Distance)

In [ ]:
# Match Q3/Q4 countries to their nearest Q1/Q2 analogue using Mahalanobis distance
# on structural features (GDP, urbanization, governance, etc.)
structural_cols = input_cols  # proxy; extend with governance/GDP if available

# Compute covariance matrix of structural features
cov_matrix = all_data[structural_cols].cov().values
cov_inv = np.linalg.pinv(cov_matrix)

# Reference set: Q1 and Q2 countries
reference_mask = quadrants.isin(["Q1", "Q2"])
target_mask = quadrants.isin(["Q3", "Q4"])

reference_countries = all_data.loc[reference_mask, structural_cols]
target_countries = all_data.loc[target_mask, structural_cols]

# Find nearest reference country for each target country
matches = []
for t_country in target_countries.index:
    t_vec = target_countries.loc[t_country].values
    min_dist = np.inf
    best_match = None
    for r_country in reference_countries.index:
        r_vec = reference_countries.loc[r_country].values
        dist = mahalanobis(t_vec, r_vec, cov_inv)
        if dist < min_dist:
            min_dist = dist
            best_match = r_country
    matches.append({
        "country": t_country,
        "quadrant": quadrants[t_country],
        "nearest_analogue": best_match,
        "analogue_quadrant": quadrants[best_match],
        "mahalanobis_dist": min_dist
    })

matches_df = pd.DataFrame(matches).sort_values("mahalanobis_dist")
print("Policy Transfer Matches (Q3/Q4 -> Q1/Q2 analogues):")
print(matches_df.head(20).to_string(index=False))

## 4. Recommendation Cards

In [ ]:
# Generate per-quadrant recommendation summaries
recommendations = {
    "Q1": {
        "label": "High-Input, High-Efficiency",
        "strategy": "Maintain & Innovate",
        "actions": [
            "Continue current resource levels with incremental efficiency gains",
            "Invest in frontier health technologies (precision medicine, AI diagnostics)",
            "Serve as mentors / knowledge exporters for Q3/Q4 countries",
            "Focus on remaining inequities within the country (subnational gaps)"
        ]
    },
    "Q2": {
        "label": "Low-Input, High-Efficiency",
        "strategy": "Scale & Protect",
        "actions": [
            "Document and formalize best practices for knowledge transfer",
            "Gradually increase resource investment to sustain gains at scale",
            "Strengthen health system resilience against shocks (pandemics, climate)",
            "Invest in preventive care and community health infrastructure"
        ]
    },
    "Q3": {
        "label": "High-Input, Low-Efficiency",
        "strategy": "Reform & Reallocate",
        "actions": [
            "Conduct waste audits (administrative overhead, duplicate services)",
            "Adopt efficiency practices from matched Q1/Q2 analogues",
            "Shift spending from tertiary to primary care and prevention",
            "Implement performance-based financing and accountability mechanisms"
        ]
    },
    "Q4": {
        "label": "Low-Input, Low-Efficiency",
        "strategy": "Invest & Build",
        "actions": [
            "Increase health expenditure with targeted external financing",
            "Build foundational infrastructure (primary care, workforce training)",
            "Adopt leapfrog technologies from Q2 best-practice analogues",
            "Strengthen governance and reduce health spending leakage/corruption"
        ]
    }
}

# Display recommendation cards
for q, rec in recommendations.items():
    n_countries = (quadrants == q).sum()
    print(f"{'=' * 60}")
    print(f"  {q}: {rec['label']}  ({n_countries} countries)")
    print(f"  Strategy: {rec['strategy']}")
    print(f"{'=' * 60}")
    for i, action in enumerate(rec["actions"], 1):
        print(f"  {i}. {action}")
    print()